## 1. Importar os dados pré-processados

In [1]:
import joblib
import pandas as pd
import os

print("=== CARREGAMENTO DOS DADOS E ARTEFATOS PRÉ-PROCESSADOS ===\n")

input_dir = "data/processed"

# 1. Carregar o dataframe processado completo (opcional, útil para referência/análises gerais)
df_encoded = pd.read_csv(os.path.join(input_dir, "telco_churn_processed.csv"))
print(f"✓ Dataframe processado carregado: {df_encoded.shape}")

# 2. Carregar os splits de treino e teste (mesmos usados no treino do modelo)
X_train = pd.read_csv(os.path.join(input_dir, "X_train.csv"))
X_test = pd.read_csv(os.path.join(input_dir, "X_test.csv"))
y_train = pd.read_csv(os.path.join(input_dir, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(input_dir, "y_test.csv")).squeeze("columns")
print(f"✓ Splits carregados — X_train: {X_train.shape}, X_test: {X_test.shape}")

# 3. Carregar o scaler ajustado no treino
scaler = joblib.load(os.path.join(input_dir, "scaler.pkl"))
print("✓ Scaler carregado")

# 4. Carregar a lista de colunas do encoding
feature_columns = joblib.load(os.path.join(input_dir, "feature_columns.pkl"))
print(f"✓ Lista de colunas carregada ({len(feature_columns)} features)")

print("\n✓ Carregamento concluído!")

=== CARREGAMENTO DOS DADOS E ARTEFATOS PRÉ-PROCESSADOS ===

✓ Dataframe processado carregado: (7043, 48)
✓ Splits carregados — X_train: (5634, 47), X_test: (1409, 47)
✓ Scaler carregado
✓ Lista de colunas carregada (47 features)

✓ Carregamento concluído!


In [4]:
import mlflow

model_uri = "runs:/f8da89fbaa0148d0afd4b254c272e515/model"
model = mlflow.sklearn.load_model(model_uri)

In [7]:
feature_columns = list(model.feature_names_in_)
print(f"Features corretas ({len(feature_columns)}): {feature_columns}")

Features corretas (18): ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_ratio', 'tenure_years', 'is_new_customer', 'charges_per_service', 'is_month_to_month', 'is_electronic_check', 'has_fiber_optic', 'has_internet', 'new_and_monthly', 'fiber_and_echeck', 'InternetService_Fiber optic', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Electronic check', 'tenure_group_49-72m']


In [8]:
print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [12]:
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate
from sklearn.metrics import accuracy_score

sensitive_feature = X_test['gender_Male'].map({0: 'Female', 1: 'Male'})

metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'false_positive_rate': false_positive_rate,
        'false_negative_rate': false_negative_rate
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

print("Métricas por grupo (gênero):")
print(metric_frame.by_group)
print("\nDiferença máxima entre grupos:")
print(metric_frame.difference())

ModuleNotFoundError: No module named 'fairlearn'